In [2]:
import torch
from torch.utils.data import Dataset

PAD_ID = 2

def flatten_state(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return torch.tensor(flat, dtype=torch.long)

class PrefixNextBitPaddedDataset(Dataset):
    def __init__(self, samples):
        self.data = []
        for seq, label in samples:
            x = flatten_state(seq)
            y = 1 if label > 0 else 0
            self.data.append((x, y))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

In [3]:
def collate_pad(batch):
    """
    batch: list of (x, y) where x is [T_i] of 0/1 tokens
    returns:
      X: [B, T] int64 in {0,1,PAD_ID}
      Y: [B]     int64 in {0,1}
      M: [B, T]  bool   True where PAD
    """
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    padded, masks = [], []
    for x in xs:
        pad_len = max_len - x.size(0)
        if pad_len > 0:
            x_pad = torch.cat([x, torch.full((pad_len,), PAD_ID, dtype=torch.long)])
            m_pad = torch.cat([torch.zeros(x.size(0), dtype=torch.bool),
                               torch.ones(pad_len, dtype=torch.bool)])   # True = PAD
        else:
            x_pad = x
            m_pad = torch.zeros_like(x, dtype=torch.bool)
        padded.append(x_pad); masks.append(m_pad)

    X = torch.stack(padded)                   # [B, T]
    Y = torch.tensor(ys, dtype=torch.long)    # [B]
    M = torch.stack(masks)                    # [B, T] (True=PAD)
    return X, Y, M

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class PolicyOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=3,         # tokens {0,1,PAD_ID}
        pad_id=PAD_ID,
        max_len=4096,
        d_model=256,
        nhead=8,
        nlayers=4,
        dim_ff=512,
        dropout=0.1,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)  # learned positional embeddings

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, 2)  # logits for {0,1}

    def forward(self, x, pad_mask):
        """
        x:        [B, T] int64 in {0,1,pad_id}
        pad_mask: [B, T] bool, True where PAD (ignored by attention)
        """
        B, T = x.size()
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)  # [B, T]
        h = self.token_emb(x) + self.pos_emb(pos)                         # [B, T, D]

        # Mask pads inside the encoder
        h = self.encoder(h, src_key_padding_mask=pad_mask)                # [B, T, D]

        # Pool from the last non-pad token per sequence
        valid = ~pad_mask                                # True where real token
        lengths = valid.long().sum(dim=1) - 1            # index of last real token
        lengths = lengths.clamp(min=0)                   # guard against all-pad (shouldn't happen)
        idx = lengths.view(B, 1, 1).expand(B, 1, h.size(-1))
        last_h = h.gather(1, idx).squeeze(1)             # [B, D]

        logits = self.head(self.norm(last_h))            # [B, 2]
        return logits

In [8]:
from torch.utils.data import DataLoader
import torch.optim as optim

def train_overfit(samples, epochs=50, batch_size=32, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ds = PrefixNextBitPaddedDataset(samples)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=collate_pad)

    model = PolicyOnlyTransformer(vocab_size=3, pad_id=PAD_ID, max_len=4096).to(device)
    opt = optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)

    for ep in range(1, epochs+1):
        model.train()
        tot_loss = tot_acc = n = 0
        for X, Y, M in loader:
            X, Y, M = X.to(device), Y.to(device), M.to(device)
            logits = model(X, pad_mask=M)
            loss = F.cross_entropy(logits, Y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            with torch.no_grad():
                acc = (logits.argmax(-1) == Y).float().sum().item()
            bs = X.size(0)
            tot_loss += loss.item() * bs
            tot_acc  += acc
            n += bs
        if ep % 2 == 0:
            print(f"epoch {ep:3d} | loss {tot_loss/n:.4f} | acc {tot_acc/n:.3f}")

    return model

In [6]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

@torch.no_grad()
def evaluate_policy(model, dataset, batch_size=256, device=None, return_mistakes=False):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

    tot_loss = tot_correct = n = 0
    mistakes = []
    for X, Y, M in loader:
        X, Y, M = X.to(device), Y.to(device), M.to(device)
        logits = model(X, pad_mask=M)            # [B,2]
        loss = F.cross_entropy(logits, Y, reduction="sum")
        pred = logits.argmax(dim=-1)             # [B]
        print(pred)

        tot_loss    += loss.item()
        tot_correct += (pred == Y).sum().item()
        n           += X.size(0)

        if return_mistakes:
            wrong = (pred != Y).nonzero(as_tuple=False).squeeze(1)
            for i in wrong.tolist():
                mistakes.append({
                    "idx": n - X.size(0) + i,
                    "true": int(Y[i].item()),
                    "pred": int(pred[i].item()),
                    "prob": float(torch.softmax(logits[i], -1)[pred[i]].item()),
                })

    return {
        "loss": tot_loss / max(n, 1),
        "acc":  tot_correct / max(n, 1),
        "n": n,
        "mistakes": mistakes if return_mistakes else None,
    }

In [6]:
from sequence_generator import generate_dataset
from sklearn.model_selection import train_test_split

## Create dataset on specified length and evaluate on that length

r = 5

original_dataset = generate_dataset(r)
for item in original_dataset:
    if len(item[0][0]) == 0:
        original_dataset.remove(item)
        break

In [7]:
def create_partial_str(seq):
    s = ""
    for i in range(4):
        cur_seq = seq[i]
        for j in range(len(cur_seq)):
            if cur_seq[j] == 1:
                s += "1"
            else:
                s += "0"
    return s

partial_strs = set()
double_seq = []
for item in original_dataset:
    seq = item[0]
    cur_str = create_partial_str(seq)
    if cur_str in partial_strs:
        double_seq.append(seq[:4])
    else:
        partial_strs.add(cur_str)

print(len(double_seq))

train_dataset, test_dataset = train_test_split(original_dataset, test_size=0.1)


5


In [8]:
count = [0, 0]
for item in train_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

count = [0, 0]
for item in test_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

model = train_overfit(train_dataset, epochs=20)
eval_dataset = PrefixNextBitPaddedDataset(test_dataset)
evaluate_policy(model, eval_dataset)

[34, 45]
[0, 9]
epoch  10 | loss 0.2577 | acc 0.848
epoch  20 | loss 0.1145 | acc 0.937
tensor([0, 1, 1, 0, 1, 1, 0, 0, 1])


/Users/arnavarora/anaconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:296: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /private/var/folders/nz/j6p8yfhx1mv_0grj5xl4650h0000gp/T/abs_1aidzjezue/croot/pytorch_1687856425340/work/aten/src/ATen/NestedTensorImpl.cpp:179.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


{'loss': 1.2173212899102106,
 'acc': 0.5555555555555556,
 'n': 9,
 'mistakes': None}

In [9]:
from sequence_generator import generate_dataset

## Create dataset on other lengths and evaluate on new length

r = 9

original_dataset = generate_dataset(r)
original_dataset.extend(generate_dataset(r - 2))
#original_dataset.extend(generate_dataset(r - 4))

for item in original_dataset:
    if len(item[0][0]) == 0:
        original_dataset.remove(item)

count = [0, 0]
for item in original_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

model = train_overfit(original_dataset, epochs=10)

[2156, 3486]
epoch   2 | loss 0.5112 | acc 0.704
epoch   4 | loss 0.4643 | acc 0.739


KeyboardInterrupt: 

In [ ]:
eval_samples = generate_dataset(r + 4)

for item in eval_samples:
    if len(item[0][0]) == 0:
        eval_samples.remove(item)

eval_dataset = PrefixNextBitPaddedDataset(eval_samples)
print("Start evaluation")

evaluate_policy(model, eval_dataset)

KeyboardInterrupt: 

In [11]:
import random

single_example = original_dataset[int(random.random() * len(original_dataset))]
print(single_example)

seq, next_bit = single_example
for s in seq:
    print(s)

# single_eval_dataset = PrefixNextBitPaddedDataset([single_example])
# evaluate_policy(model, single_eval_dataset)

([(1, 1, -1, 1, -1), (1, 1, 1, -1), (1, -1, -1, 1), (1, -1, -1, 1)], 1)
(1, 1, -1, 1, -1)
(1, 1, 1, -1)
(1, -1, -1, 1)
(1, -1, -1, 1)


In [13]:
from cnf_automator import generate_encoding, verify_assignment
import json

def conversion(s):
    s = s.replace('−', '-')
    s = s.split("\n")[1:-1]
    seqs = [seq[3:] for seq in s]
    new_seqs = [json.loads(seq) for seq in seqs]
    return new_seqs

s = """
X: [1, -1, 1, 1, -1, -1, 1, 1, 1, -1, -1]
Y: [-1, 1, 1, -1, 1, -1, -1, 1, -1, 1, 1]
Z: [1, 1, -1, 1, -1, 1, -1, -1, 1, 1, -1]
W: [1, -1, -1, 1, 1, -1, 1, -1, -1, 1, 1]
"""

r = 23
seq = conversion(s)
cnf, varmap, pmap_lags, pool = generate_encoding(r)
sat = verify_assignment(seq, varmap, cnf)
sat

False

In [9]:
r = 23
cnf, varmap, pmap_lags, pool = generate_encoding(r)

def save_encoding(cnf, varmap, pmap_lags, pool, filename):
    cnf.to_file(filename)
    meta_filename = filename.rsplit('.', 1)[0] + "_meta.json"

    meta = {
        "varmap": varmap,
        "pmap_lags": {str(k): v for k, v in pmap_lags.items()},
        "idmap": pool.obj2id,  # maps variable name → integer ID
        "revmap": {str(v): k for k, v in pool.obj2id.items()}  # integer ID → variable name
    }

    with open(meta_filename, "w") as f:
        json.dump(meta, f, indent=2)

save_encoding(cnf, varmap, pmap_lags, pool, f"encoding_{r}.cnf")

In [ ]:
import json
from pathlib import Path
from pysat.formula import CNF, IDPool


def _rebuild_pool_from_idmap(idmap):
    """
    Reconstruct an IDPool whose IDs match exactly the saved mapping.
    We replay IDs in ascending order so that pool.id(name) assigns
    the intended integer each time.
    """
    pairs = sorted(((int(v), k) for k, v in idmap.items()), key=lambda x: x[0])
    pool = IDPool(start_from=1)
    for expected_id, name in pairs:
        got = pool.id(name)
        if got != expected_id:
            raise RuntimeError(f"Rebuild mismatch for {name}: expected {expected_id}, got {got}")
    return pool


def load_turyn_instance(cnf_path):
    """
    Load CNF + metadata and rebuild IDPool and mappings.
    Returns: (cnf, varmap, pmap_lags, pool, meta)
    """
    cnf_path = Path(cnf_path)
    if not cnf_path.exists():
        raise FileNotFoundError(cnf_path)

    # Load CNF (DIMACS)
    cnf = CNF(from_file=str(cnf_path))

    # Load metadata JSON
    meta_path = cnf_path.with_name(cnf_path.stem + "_meta.json")
    if not meta_path.exists():
        # Also allow direct path to JSON metadata
        if cnf_path.suffix == ".json":
            meta_path = cnf_path
        else:
            raise FileNotFoundError(meta_path)

    meta = json.loads(meta_path.read_text())

    # Rebuild pool from saved idmap
    idmap = meta.get("idmap") or {}
    pool = _rebuild_pool_from_idmap(idmap)

    # Extract auxiliary maps
    varmap = meta.get("varmap", {})
    pmap_lags = {int(k): v for k, v in (meta.get("pmap_lags") or {}).items()}

    return cnf, varmap, pmap_lags, pool, meta